In [1]:
USER = "varom"

import ROOT
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix
import os 
import seaborn as sns

path_root = '/data/at3/common/EWK2L3L_ANA-SUSY-2024-02/FFntuples/RUN2_v2'

Samples are: VVV, VV_Sh2214, Vy_Sh2214, Wjets_lv, Zjets_ll, Zjets_lowMLL, higgs, othertop, singletop_lep, ttbar_allhad, ttbar_nonallhad, ttbar_dil

# 1. Open yml file

In [ ]:
import yaml

def open_yml(yml_file): #opens yml file
    with open(yml_file) as f:
        samples = yaml.safe_load(f)["samples"]
    
    return samples


def get_sample(samples_yml, sample_name): # auxiliary function that allows to check if the specific sample is inside the yml file
    for sample in samples_yml:
        if sample["name"] == sample_name:
            return sample_name
    raise ValueError(f"Sample '{sample_name}' not found.")

def get_variable(variables_yml, variable_name):
    for variable in variables_yml:
        if variable["name"] == variable_name:
            return variable
    raise ValueError(f"Variable '{variable_name}' not found.")

In [ ]:
samples_yml = open_yml('samples.yml')
variables_yml = open_yml('variables.yml')

# 2. Print/Check the names of the .root files for each sample

In [ ]:
from pathlib import Path

def get_root_files(samples_yml,sample_name,path_root):

    path_root = Path(path_root)

    # Buscar el sample
    sample = next(
        (s for s in samples_yml if s["name"] == sample_name),
        None
    )

    if sample is None:
        raise ValueError(f"Sample '{sample_name}' not found.")

    files = []

    for dsid in sample["dsids"]:
        for campaign in sample["campaigns"]:

            filename = (
                f"{sample_name}_{dsid}_{campaign}_"
                f"{sample['simulation_type']}.root"
            )

            filepath = path_root / filename

            if filepath.exists():
                files.append(str(filepath))
            else:
                print(f"WARNING: {filepath} no existe.")

    return files

def print_root_files(samples_yml,sample_name,path_root):

    files = get_root_files(samples_yml,sample_name,path_root)

    print(f'FILES FOUND FOR THE SAMPLE {sample_name} ARE:')

    i=0

    for file in files:
        print(file)
        i = i+1
    
    print(f'NUMBER OF FILES FOR THE SAMPLE {sample_name} ARE {i} FILES')

In [ ]:
#print_root_files('VV_Sh214',path_root)

FILES FOUND FOR THE SAMPLE VV ARE:
NUMBER OF FILES FOR THE SAMPLE VV ARE 0 FILES


# 3. Build histograms without plotting

In [ ]:
def get_binning(variables_yml, variable_name):
    variable_info = next((v for v in variables_yml if v["name"] == variable_name), None)
    if variable_info is None:
        raise ValueError(f"{variable_name} variable not found.")
    binning = variable_info["binning"]
    bins = binning["number_of_bins"]
    xrange = (binning["min"], binning["max"])
    return bins, xrange

In [ ]:
from hist import Hist
import uproot
import awkward as ak

def make_histograms(samples_yml, sample_name, variables_yml, variable_names, path_root,
                        weight="weight_total_NOSYS", selection=None,tree_name="analysis", step_size="100 MB"):

    files = get_root_files(samples_yml, sample_name, path_root)

    hists = {}
    for variable_name in variable_names:
        bins, xrange = get_binning(variables_yml, variable_name)
        hists[variable_name] = Hist.new.Reg(bins, xrange[0], xrange[1], name=variable_name).Weight()

    branches = list(dict.fromkeys(list(variable_names) + [weight]))  # sin duplicados, por si weight coincide con alguna variable

    for events in uproot.iterate(files, tree_name, expressions=branches, library="ak", step_size=step_size):

        if selection is not None:
            mask = selection(events)
            weights = events[weight][mask]
        else:
            mask = None
            weights = events[weight]

        for variable_name in variable_names:
            values = events[variable_name][mask] if mask is not None else events[variable_name]
            hists[variable_name].fill(values, weight=weights)

    return hists

In [ ]:
def make_background_histograms(samples_yml, variables_yml, variable_names, path_root, selection=None):

    results = {variable_name: {"samples": {}, "total": None} for variable_name in variable_names}

    for sample in samples_yml:

        if sample["type"] != "background":
            continue

        print(f"Building histograms for {sample['name']}...")

        hists = make_histograms_multi(samples_yml, sample["name"], variables_yml, variable_names,
                                       path_root, selection=selection)

        for variable_name in variable_names:

            h = hists[variable_name]

            results[variable_name]["samples"][sample["name"]] = {
                "hist": h,
                "label": sample["label"],
                "color": sample["color"],
                "type": sample["type"],
                "dsids": sample["dsids"],
                "campaigns": sample["campaigns"],
                "simulation_type": sample["simulation_type"],
            }

            if results[variable_name]["total"] is None:
                results[variable_name]["total"] = h.copy()
            else:
                results[variable_name]["total"] += h

    return results

# 4. Plot stack

In [ ]:
def plot_stack(background, variables_yml, variable_name, ax, output=None):

    hist_list = []
    color_list = []
    process_label_list = []

    for sample in background["samples"].values():
        hist_list.append(sample["hist"])
        color_list.append(sample["color"])
        process_label_list.append(sample["label"])

    hep.histplot(hist_list, stack=True, histtype="fill", color=color_list, label=process_label_list, ax=ax)

    variable_info = next((v for v in variables_yml if v["name"] == variable_name), None)

    if variable_info is None:
        raise ValueError(f"{variable_name} variable not found.")

    if variable_info["units"] is not None:
        xlabel = variable_info["name"].replace("_NOSYS", "")
        xlabel = f"{xlabel}_{variable_info['units']}"
    else:
        xlabel = variable_info["name"].replace("_NOSYS", "")

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Events")
    ax.set_title(variable_info["name"])

    if variable_info["logy"] == True:
        ax.set_yscale("log")

    ax.legend(fontsize=8)

    if output is not None:
        ax.figure.savefig(output, dpi=300, bbox_inches="tight")

In [ ]:
import mplhep as hep

hep.style.use("ATLAS")

def subplot_stack(samples_yml, variables_yml, path_root, title, selection=None):

    variable_names = [var["name"] for var in variables_yml]

    all_bkg = make_background_histograms(samples_yml, variables_yml, variable_names, path_root, selection=selection)

    N = len(variable_names)
    ncols = 3
    nrows = (N + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 4*nrows))
    fig.suptitle(title, fontsize=24)
    axes = axes.flatten()

    for ax, variable_name in zip(axes, variable_names):
        plot_stack(all_bkg[variable_name], variables_yml, variable_name, ax=ax)

    for ax in axes[N:]:
        ax.remove()

    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.show()
    return fig, axes

In [ ]:
def save_plot(directory, fig, pdfname):

    # USER
    # directory: directory into which the pdf file is going
    # fig: figure
    # pdfname
    
    os.makedirs(directory, exist_ok=True)

    filename = os.path.join(directory, f"{pdfname}.pdf")
    fig.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close(fig)